# Feature engineering

In [20]:
import pandas as pd

daily_sales = pd.read_csv(
    "../data/processed/daily_sales.csv",
    parse_dates=["Date"]
)

print(daily_sales.head())
print(daily_sales.dtypes)

        Date   Revenue  Quantity  Orders  Customers
0 2009-12-01  54351.23     26098     119         91
1 2009-12-02  63172.58     31804     115         94
2 2009-12-03  73972.45     49221     124        106
3 2009-12-04  40582.32     21210      89         76
4 2009-12-05   9803.05      5119      30         26
Date         datetime64[ns]
Revenue             float64
Quantity              int64
Orders                int64
Customers             int64
dtype: object


### **Feature 1 - Time features**

In [21]:
daily_sales["Year"] = daily_sales["Date"].dt.year
daily_sales["Month"] = daily_sales["Date"].dt.month
daily_sales["DayOfWeek"] = daily_sales["Date"].dt.dayofweek

### **Feature 2 - Weekend indicator**

In [22]:
daily_sales["IsWeekend"] = (
    daily_sales["DayOfWeek"] >= 5
).astype(int)

### **Lag Features**

In [23]:
daily_sales["Lag_1"] = (
    daily_sales["Revenue"].shift(1)
)

### **Previous 7 day revenue**

In [24]:
daily_sales["Lag_7"] = (
    daily_sales["Revenue"].shift(7)
)

### **Previous 14 day revenue**

In [25]:
daily_sales["Lag_14"] = (
    daily_sales["Revenue"].shift(14)
)

### **Rolling features**

In [26]:
daily_sales["RollingMean_7"] = (
    daily_sales["Revenue"]
    .shift(1)
    .rolling(7)
    .mean()
)

daily_sales["RollingMean_14"] = (
    daily_sales["Revenue"]
    .shift(1)
    .rolling(14)
    .mean()
)

daily_sales["RollingMean_28"] = (
    daily_sales["Revenue"]
    .shift(1)
    .rolling(28)
    .mean()
)

In [27]:
print(daily_sales.columns.tolist())

['Date', 'Revenue', 'Quantity', 'Orders', 'Customers', 'Year', 'Month', 'DayOfWeek', 'IsWeekend', 'Lag_1', 'Lag_7', 'Lag_14', 'RollingMean_7', 'RollingMean_14', 'RollingMean_28']


### **Forecast dataset**

In [28]:
forecast_df = daily_sales[
    [
        "Date",
        "Year",
        "Month",
        "DayOfWeek",
        "IsWeekend",
        "Lag_1",
        "Lag_7",
        "Lag_14",
        "RollingMean_7",
        "Revenue"
    ]
].copy()

forecast_df = (
    forecast_df
    .dropna()
    .reset_index(drop=True)
)

forecast_df.head()

,Date,Year,Month,DayOfWeek,IsWeekend,Lag_1,Lag_7,Lag_14,RollingMean_7,Revenue
0,2009-12-16,2009,12,2,0,50262.29,49476.23,54351.23,45576.697143,52545.55
1,2009-12-17,2009,12,3,0,52545.55,40265.66,63172.58,46015.171429,30638.25
2,2009-12-18,2009,12,4,0,30638.25,44233.96,73972.45,44639.827143,42470.29
3,2009-12-20,2009,12,6,1,42470.29,39447.89,40582.32,44387.874286,11382.54
4,2009-12-21,2009,12,0,0,11382.54,22176.46,9803.05,40378.538571,16322.95


In [29]:
forecast_df = (
    forecast_df
    .dropna()
    .reset_index(drop=True)
)

In [30]:
print("Forecasting dataset shape:", forecast_df.shape)

print("\nColumns:")
print(forecast_df.columns.tolist())

print("\nMissing values:")
print(forecast_df.isnull().sum())

forecast_df.head()

Forecasting dataset shape: (590, 10)

Columns:
['Date', 'Year', 'Month', 'DayOfWeek', 'IsWeekend', 'Lag_1', 'Lag_7', 'Lag_14', 'RollingMean_7', 'Revenue']

Missing values:
Date             0
Year             0
Month            0
DayOfWeek        0
IsWeekend        0
Lag_1            0
Lag_7            0
Lag_14           0
RollingMean_7    0
Revenue          0
dtype: int64


,Date,Year,Month,DayOfWeek,IsWeekend,Lag_1,Lag_7,Lag_14,RollingMean_7,Revenue
0,2009-12-16,2009,12,2,0,50262.29,49476.23,54351.23,45576.697143,52545.55
1,2009-12-17,2009,12,3,0,52545.55,40265.66,63172.58,46015.171429,30638.25
2,2009-12-18,2009,12,4,0,30638.25,44233.96,73972.45,44639.827143,42470.29
3,2009-12-20,2009,12,6,1,42470.29,39447.89,40582.32,44387.874286,11382.54
4,2009-12-21,2009,12,0,0,11382.54,22176.46,9803.05,40378.538571,16322.95


In [31]:
forecast_df.to_csv(
    "../data/processed/forecast_dataset.csv",
    index=False
)